# 📘 AI-Powered Predictive Maintenance: Deep Dive
**Project:** Remaining Useful Life (RUL) Estimation using LSTM.  
**Objective:** Build a robust deep learning model to predict when a machine will fail based on sensor data.

---

### 1️⃣ Import Libraries
Here we import all necessary libraries.
* `torch`: The PyTorch deep learning framework.
* `pandas/numpy`: For data manipulation.
* `sklearn`: For metrics (R2 Score) and data scaling.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import pickle

### 2️⃣ Device Configuration (GPU Check)
This cell checks if a NVIDIA GPU is available. Using a GPU significantly speeds up LSTM training. If not found, it falls back to CPU.

In [ ]:
if torch.cuda.is_available():
    print("=== CUDA GPUs Found ===")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i} -> {torch.cuda.get_device_name(i)}")
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

print(f"✅ Using Device: {device}")

=== CUDA GPUs Found ===
GPU 0 -> Tesla T4
✅ Using Device: cuda:0


### 3️⃣ Load Dataset
Reading the CSV file containing sensor data. We also convert the `timestamp` column to a proper datetime object to ensure correct ordering.

In [ ]:
# Please verify the file path before running
file_path = "/content/drive/MyDrive/preprocessed_smart_data.csv"
df = pd.read_csv(file_path)

# Convert timestamp to datetime object
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("Data Loaded Successfully.")
df.head()

Data Loaded Successfully.


,timestamp,machine_id,temperature,vibration,humidity,pressure,energy_consumption,machine_status,anomaly_flag,predicted_remaining_life,...,pressure_diff,vibration_zscore,temp_zscore,vibration_ma,vibration_std,temperature_roc,vibration_roc,pressure_roc,humidity_roc,energy_consumption_roc
0,2025-01-01 01:37:00,1,62.59,48.08,60.13,1.62,4.16,1,0,363,...,NaN,-0.128943,-1.238613,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 01:42:00,1,70.88,68.67,50.57,1.05,2.36,1,0,111,...,-0.57,1.245057,-0.412248,NaN,NaN,8.29,20.59,-0.57,-9.56,-1.80
2,2025-01-01 02:13:00,1,62.34,72.68,60.47,2.24,4.95,1,0,192,...,1.19,1.512650,-1.263534,NaN,NaN,-8.54,4.01,1.19,9.90,2.59
3,2025-01-01 02:19:00,1,88.03,53.36,50.79,3.08,2.25,1,0,301,...,0.84,0.223399,1.297301,NaN,NaN,25.69,-19.32,0.84,-9.68,-2.70
4,2025-01-01 03:05:00,1,79.39,49.44,56.99,2.54,2.76,1,0,127,...,-0.54,-0.038188,0.436047,NaN,NaN,-8.64,-3.92,-0.54,6.20,0.51


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 4️⃣ Feature Engineering: Moving Averages
Sensors (like vibration) can be noisy.
* **Why?** Rolling averages smooth out short-term fluctuations and highlight longer-term trends.
* **How?** We calculate the mean of the last 5 readings.

In [ ]:
window = 5

# Calculate rolling mean for Temperature and Vibration per machine
df['temp_moving_avg'] = df.groupby('machine_id')['temperature'].rolling(window=window).mean().reset_index(0, drop=True)
df['vib_moving_avg'] = df.groupby('machine_id')['vibration'].rolling(window=window).mean().reset_index(0, drop=True)

### 5️⃣ Feature Engineering: Rate of Change
* **Why?** Sudden spikes (sharp changes) are often more indicative of failure than the absolute value itself.
* **How?** We calculate the difference between the current value and the previous value.

In [ ]:
df['temp_rate_change'] = df.groupby('machine_id')['temperature'].diff().fillna(0)
df['vib_rate_change'] = df.groupby('machine_id')['vibration'].diff().fillna(0)

# Drop NaN values created by the rolling window
df = df.dropna()

print(f"Feature Engineering Complete. New Shape: {df.shape}")

Feature Engineering Complete. New Shape: (98323, 36)


### 6️⃣ Feature Selection
Defining exactly which columns will be used as inputs (X) for the model.

In [ ]:
features = [
    'temperature', 'vibration', 'humidity', 'pressure', 'energy_consumption',
    'temp_moving_avg', 'vib_moving_avg', 'temp_rate_change', 'vib_rate_change',
]

print(f"Selected Features ({len(features)}): {features}")

Selected Features (9): ['temperature', 'vibration', 'humidity', 'pressure', 'energy_consumption', 'temp_moving_avg', 'vib_moving_avg', 'temp_rate_change', 'vib_rate_change']


### 7️⃣ Helper Function: Data Preparation Pipeline
This function automates the complex steps:
1. **Imbalance Handling:** It identifies "Low RUL" cases (machines about to fail) and oversamples them.
2. **Noise Injection:** Adds slight random noise to synthetic samples to prevent overfitting.
3. **Scaling:** Standardizes data (Mean=0, Std=1) for better LSTM convergence.
4. **Sequencing:** Converts data into 3D arrays `(Samples, Time Steps, Features)` required by LSTM.

In [ ]:
def prepare_data(df, seq_len=20):
    """
    Prepares time-series sequences for LSTM WITHOUT oversampling or leakage.
    Scaling is expected to be done OUTSIDE this function.
    """

    X = df[features].values
    y = df['predicted_remaining_life'].values

    X_seq, y_seq = [], []

    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len])

    return np.array(X_seq), np.array(y_seq)

SEQ_LEN = 20
EPOCHS = 50
BATCH_SIZE = 64

### 8️⃣ Helper Function: DataLoader Creation
Splits data into Training and Testing sets and wraps them in PyTorch DataLoaders for batch processing.

In [ ]:
def get_dataloaders(train_X, train_y, batch_size=64):
  train_df, test_df = train_test_split(df, test_size=0.2, shuffle=False)

# Split زمني صحيح
train_df, test_df = train_test_split(df, test_size=0.2, shuffle=False)

# Scaling على train فقط
scaler = StandardScaler()
train_df[features] = scaler.fit_transform(train_df[features])
test_df[features]  = scaler.transform(test_df[features])

# تكوين Sequences
X_train, y_train = prepare_data(train_df, seq_len=SEQ_LEN)
X_test, y_test   = prepare_data(test_df, seq_len=SEQ_LEN)

     """
     Creates DataLoader ONLY for training data.
     Test data should already be separated and prepared.
    """

    train_X = torch.tensor(train_X, dtype=torch.float32).to(device)
    train_y = torch.tensor(train_y, dtype=torch.float32).unsqueeze(1).to(device)

    train_loader = DataLoader(
        TensorDataset(train_X, train_y),
        batch_size=batch_size,
        shuffle=True
    )

    return train_loader

### 9️⃣ Model Architecture: LSTM
Defining the Deep Learning model structure.
* **LSTM Layers:** To capture time-dependent patterns.
* **Dropout:** To prevent overfitting.
* **Fully Connected (FC) Layer:** To output the final single value (RUL).

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size=len(features), hidden_size=200, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])

---
## 🔟 Final Model Training
We skip the grid search here (as we already found the best parameters) and proceed directly to training the optimal model.

**Optimal Hyperparameters:**
* `Threshold`: 80
* `Momentum`: 0.8
* `Sequence Length`: 20
* `Epochs`: 50

In [ ]:
# Configuration
FINAL_THRESHOLD = 80
FINAL_MOMENTUM = 0.8
SEQ_LEN = 20
EPOCHS = 50

print("Preparing Final Data...")
# 1. Prepare Data
X_seq, y_seq, final_scaler = prepare_data(df, threshold=FINAL_THRESHOLD, seq_len=SEQ_LEN)
train_loader, X_test, y_test = get_dataloaders(X_seq, y_seq)

# 2. Initialize Model
model = LSTMRegressor().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=FINAL_MOMENTUM)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.8, patience=5, min_lr=1e-5)
criterion = nn.L1Loss() # MAE Loss

print("Model Initialized.")

Preparing Final Data...
Model Initialized.


### 1️⃣1️⃣ Training Loop
This loop iterates through the dataset `EPOCHS` times.
1. **Train:** Updates weights based on loss.
2. **Evaluate:** Checks performance on unseen Test data.
3. **Save Best:** Automatically saves the model weights if the R2 score improves.

In [ ]:
best_r2_ever = -float('inf')
best_epoch_state = None

print("Starting Training...\n")

for epoch in range(EPOCHS):
    # --- Training Phase ---
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # --- Evaluation Phase ---
    model.eval()
    with torch.no_grad():
        preds = model(X_test)
        r2_eval = r2_score(y_test.cpu().numpy(), preds.cpu().numpy())

    # Update Learning Rate based on R2
    scheduler.step(r2_eval)

    # Save Best Model Logic
    if r2_eval > best_r2_ever:
        best_r2_ever = r2_eval
        best_epoch_state = model.state_dict().copy()
        print(f"⭐ Epoch {epoch+1:02d} | New Best R2: {r2_eval:.4f} | Loss: {avg_loss:.4f}")
    else:
        print(f"   Epoch {epoch+1:02d} |            R2: {r2_eval:.4f} | Loss: {avg_loss:.4f}")

Starting Training...

⭐ Epoch 01 | New Best R2: -0.2931 | Loss: 118.2059
⭐ Epoch 02 | New Best R2: -0.2923 | Loss: 112.8885
⭐ Epoch 03 | New Best R2: 0.1420 | Loss: 107.7369
⭐ Epoch 04 | New Best R2: 0.4390 | Loss: 81.4603
⭐ Epoch 05 | New Best R2: 0.4451 | Loss: 78.7242
   Epoch 06 |            R2: 0.4450 | Loss: 78.5674
⭐ Epoch 07 | New Best R2: 0.4452 | Loss: 78.5059
⭐ Epoch 08 | New Best R2: 0.4452 | Loss: 78.4820
⭐ Epoch 09 | New Best R2: 0.4455 | Loss: 78.5018
   Epoch 10 |            R2: 0.4452 | Loss: 78.4577
   Epoch 11 |            R2: 0.4454 | Loss: 78.4597
⭐ Epoch 12 | New Best R2: 0.4455 | Loss: 78.4546
⭐ Epoch 13 | New Best R2: 0.4457 | Loss: 78.4445
   Epoch 14 |            R2: 0.4456 | Loss: 78.4411
   Epoch 15 |            R2: 0.4455 | Loss: 78.4265
   Epoch 16 |            R2: 0.4456 | Loss: 78.4685
   Epoch 17 |            R2: 0.4453 | Loss: 78.4553
   Epoch 18 |            R2: 0.4454 | Loss: 78.4574
   Epoch 19 |            R2: 0.4420 | Loss: 78.4208
   Epoch 20 |  

### 1️⃣2️⃣ Saving Results
Finally, we save the trained model weights and the scaler. This is crucial for using the model later in the Streamlit app.

In [ ]:
print("\n======================================")
print(f"🏆 FINAL RESULT | Best R2: {best_r2_ever:.4f}")
print("======================================")

if best_epoch_state:
    # Save Model
    torch.save(best_epoch_state, "Final_Best_LSTM_Model.pth")
    print("✅ Model weights saved to 'Final_Best_LSTM_Model.pth'")

    # Save Scaler
    with open("scaler.pkl", "wb") as f:
        pickle.dump(final_scaler, f)
    print("✅ Scaler saved to 'scaler.pkl'")
else:
    print("❌ No model saved.")


🏆 FINAL RESULT | Best R2: 0.4457
✅ Model weights saved to 'Final_Best_LSTM_Model.pth'
✅ Scaler saved to 'scaler.pkl'
